<a href="https://colab.research.google.com/github/Sandyo-O/FamilyCare_Agent/blob/main/%E3%80%90%E6%9C%9F%E6%9C%AB%E5%B0%88%E6%A1%88%E3%80%91%E7%94%A8RAG%E6%89%93%E9%80%A0AI%E5%A1%94%E7%BE%85%E5%8D%A0%E5%8D%9C%E5%B8%AB%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 0. 流程說明:
Step 1:使用者選擇占卜設定(占卜主題、牌陣、占卜師風格)

Step 2:使用者輸入想問的問題

Step 3:系統依據使用者選擇的牌陣隨機抽牌

Step 4:生成對應的占卜結果及建議

In [ ]:
URL="https://drive.google.com/uc?export=download&id=1ZpA_kFNc3xCYErsHcxFWrFNH6GD53rck"
!wget -O tarot_db.zip "$URL"

--2026-03-29 16:04:31--  https://drive.google.com/uc?export=download&id=1ZpA_kFNc3xCYErsHcxFWrFNH6GD53rck
Resolving drive.google.com (drive.google.com)... 74.125.20.102, 74.125.20.101, 74.125.20.138, ...
Connecting to drive.google.com (drive.google.com)|74.125.20.102|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1ZpA_kFNc3xCYErsHcxFWrFNH6GD53rck&export=download [following]
--2026-03-29 16:04:31--  https://drive.usercontent.google.com/download?id=1ZpA_kFNc3xCYErsHcxFWrFNH6GD53rck&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.135.132, 2607:f8b0:400e:c01::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.135.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 94641 (92K) [application/octet-stream]
Saving to: ‘tarot_db.zip’

tarot_db.zip        100%[===================>]  92.42K  --.-KB/s    in 0.

In [ ]:
!unzip tarot_db.zip

Archive:  tarot_db.zip
replace tarot_db/index.faiss? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

### 1. 安裝並引入必要套件

In [ ]:
!pip install -U langchain langchain-community sentence-transformers faiss-cpu gradio openai langchain-openai

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain

In [ ]:
from openai import OpenAI
import gradio as gr

### 2. 自訂 E5 embedding 類別

In [ ]:
class CustomE5Embedding(HuggingFaceEmbeddings):
    def embed_documents(self, texts):
        texts = [f"passage: {t}" for t in texts]
        return super().embed_documents(texts)

    def embed_query(self, text):
        return super().embed_query(f"query: {text}")

3. 載入 `tarot_db`

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings  # 或你自定義的 CustomE5Embedding
embedding_model = CustomE5Embedding(model_name="intfloat/multilingual-e5-small")
db = FAISS.load_local("tarot_db", embedding_model, allow_dangerous_deserialization=True)
retriever = db.as_retriever()

### 4. 設定好我們要的 LLM

In [ ]:
import os
from google.colab import userdata

如之前, 我們會用 OpenAI API。這裡使用 Groq 服務, 可改成你要的服務。

In [ ]:
api_key = userdata.get('Groq')

In [ ]:
os.environ["OPENAI_API_KEY"] = api_key

這裡的模型和 `base_url` 是用 Groq, 如果用其他服務請自行修改。

In [ ]:
model = "llama3-70b-8192"
base_url="https://api.groq.com/openai/v1"

In [ ]:
client = OpenAI(
    base_url=base_url # 使用 OpenAI 本身不需要這段
)

### 5. `prompt` 設計

In [ ]:
# 占卜主題
topics = {
    "不限主題": "不限定任何範圍，讓塔羅牌自由傳達當下最重要的命運訊息。",
    "感情": "關於戀愛、曖昧、感情走向、復合、桃花運、婚姻等情感相關議題。",
    "工作": "針對職場、人際合作、職涯發展、升遷、跳槽、創業等職業相關議題。",
    "學業": "關於考試運、課業表現、升學、選系、留學等學習相關議題。",
    "人際": "朋友、家人、同事、人際互動、人緣、人際困擾等相關議題。",
    "人生目標": "針對人生方向、自我探索、目標訂定、階段任務等整體命運走向。"
}


# 牌陣方式
spreads = {
    "單張占卜": "抽一張牌，快速針對單一問題獲取命運訊息與建議，適合臨時直覺占卜。",
    "三張牌": "抽三張牌，分別代表『過去』『現在』『未來』，幫助看清問題脈絡與未來可能發展。",
    "愛情十字陣": "五張牌排列成十字，專門針對感情問題設計，洞察雙方現狀、潛意識、阻礙、建議與未來走向。",
    "五芒星陣": "五張牌排列成五芒星，適用於複雜問題，分別代表現況、障礙、潛能、建議、結果，全面分析局勢。"
}


# 占卜師風格（不同風格對應不同 system prompt）
styles = {
    "冷靜分析型": "你是冷靜理性的命運分析者，以條理清晰、客觀務實方式解讀塔羅，避免誇張，重點在於實用建議。⚠️請**完全使用繁體中文**作答，禁用任何其他語言，每次回覆務必**直擊問題核心，解讀牌義並給出清楚、具體、實際命運建議**，避免冗長空泛。",
    "溫柔療癒型": "你是溫柔療癒的命運守護者，以溫暖柔和、安慰人心的語氣，細膩解讀塔羅，給予心靈安撫與希望。⚠️請**完全使用繁體中文**作答，禁用任何其他語言，每次回覆務必**直擊問題核心，解讀牌義並給出清楚、具體、實際命運建議**，避免冗長空泛。",
    "中二魔幻型": "你是神祕中二魔幻風格占卜師，語氣戲劇化誇張，充滿神祕符號與儀式感，讓人仿若置身魔法世界。⚠️請**完全使用繁體中文**作答，禁用任何其他語言，每次回覆務必**直擊問題核心，解讀牌義並給出清楚、具體、實際命運建議**，避免冗長空泛。",
    "搞笑活潑型": "你是幽默風趣、搞笑活潑的塔羅占卜師，用輕鬆詼諧方式解讀命運，帶給旅人快樂與正能量。⚠️請**完全使用繁體中文**作答，禁用任何其他語言，每次回覆務必**直擊問題核心，解讀牌義並給出清楚、具體、實際命運建議**，避免冗長空泛。"
}

# 模型選擇
models = {
    "GPT-4": "由 OpenAI 提供的進階生成式模型，語言流暢、邏輯清晰，適合需要精準且深入解讀的占卜回應。",
    "GPT-3.5": "由 OpenAI 提供的穩定版本，生成速度快，適合一般占卜需求，文字自然且具備良好語意連貫性。",
    "Gemini": "由 Google 開發的多模態模型，表達風格偏向中性且清晰，適合結合圖像與語意做更進階應用。",
    "LLaMA 3": "Meta 開發的開源大型語言模型，搭配本地部署或 Ollama 使用，具實驗性與可調整性，適合客製化風格探索。",
    "Mistral": "輕量高效的開源模型，擅長快速生成，語氣多樣，適合搞笑風格或快速占卜回覆場景。",
    "HuggingFace": "使用 Hugging Face Transformers 架構的開源模型，如 Zephyr、Phi-3 等，可依需求部署於 Colab 或 GPU 環境。"
}


### 6. 使用 RAG 來回應

搜尋與使用者問題相關的資訊，根據我們的 prompt 樣版去讓 LLM 回應。

In [ ]:
chat_history = []

def chat_with_rag(user_input, topic, spread, style, model_name):
    global chat_history

    # 取得資料庫塔羅牌義
    docs = retriever.get_relevant_documents(user_input)
    retrieved_chunks = "\n\n".join([doc.page_content for doc in docs])

    # 風格 prompt 組合
    selected_style_prompt = styles[style]

    final_system_prompt = (
        f"你是命運之塔的塔羅占卜大師，擅長以不同占卜師風格，"
        f"針對塔羅問題進行專業解讀。\n"
        f"本次占卜主題：「{topic}」，牌陣：「{spread}」，占卜師風格：「{style}」。\n\n"
        f"{selected_style_prompt}\n\n"
        "⚠️請**完全使用繁體中文**回覆，嚴禁出現任何英文、符號或其他語言，"
        "每次回覆務必依據資料庫塔羅牌義，給予清楚、具體、實用命運建議，搭配相應語氣及符號。"
    )

    # 使用者輸入與 retrieved chunks 組合成 prompt
    final_prompt = f"""
🃏✦ 命運占卜儀式 啟動 ✦🃏

占卜主題：{topic}
牌陣方式：{spread}

依據命運之塔內典藏資料：
{retrieved_chunks}

🔮 旅人所求之命運問題為：
{user_input}

占卜師風格：{style}
"""

    ### 模型選擇邏輯 ###
    if model_name in ["GPT-4", "GPT-3.5"]:
        model_used = "gpt-4" if model_name == "GPT-4" else "gpt-3.5-turbo"
        response = client.chat.completions.create(
            model=model_used,
            messages=[
                {"role": "system", "content": final_system_prompt},
                {"role": "assistant", "content": "⚠️提醒：以下所有回覆皆需完全以繁體中文呈現，嚴禁英文及其他語言。"},
                {"role": "user", "content": final_prompt},
            ]
        )
        answer = response.choices[0].message.content

    elif model_name == "Gemini":
        response = gemini_client.generate_content([
            final_system_prompt,
            final_prompt
        ])
        answer = response.text

    elif model_name in ["LLaMA 3", "Mistral"]:
        import requests
        model_used = "llama3" if model_name == "LLaMA 3" else "mistral"
        ollama_payload = {
            "model": model_used,
            "messages": [
                {"role": "system", "content": final_system_prompt},
                {"role": "user", "content": final_prompt}
            ]
        }
        r = requests.post("http://localhost:11434/api/chat", json=ollama_payload)
        answer = r.json()["message"]["content"]

    elif model_name == "HuggingFace":
        from transformers import pipeline
        full_prompt = final_system_prompt + "\n\n" + final_prompt
        response = hf_pipe(
            full_prompt,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.95,
            eos_token_id=hf_tokenizer.eos_token_id
        )
        answer = response[0]['generated_text'].replace(full_prompt, "").strip()

    else:
        answer = "❌ 無法識別選擇的模型，請重新選擇。"

    chat_history.append((user_input, answer))
    return answer


### 7. 用 Gradio 打造 Web App

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# 🃏命運之塔✦AI塔羅占卜大師")

    # 聊天視窗
    chatbot = gr.Chatbot(
        value=[("命運之塔✦AI塔羅占卜大師",
                "🃏✦ 歡迎踏入命運之塔，旅人！✦🃏\n\n請在心中**默念你想詢問的問題三次**，當心緒靜定，便可將你的疑問刻於下方之命運石板，我將為你揭示命運答案 🔮✨")],
        height=500
    )

    msg = gr.Textbox(placeholder="請在此輸入你的人生疑問...")

    # 選單區
    with gr.Row():
        topic_dropdown = gr.Dropdown(topics, label="🔮 占卜主題")
        spread_dropdown = gr.Dropdown(spreads, label="🃏 牌陣")
        style_dropdown = gr.Dropdown(list(styles.keys()), label="✨ 占卜師風格")
        model_dropdown = gr.Dropdown(list(models.keys()), label="🧠 使用模型")

    # 當文字送出時呼叫 RAG 回應
    def respond_v2(user_input, topic, spread, style, model_name, chat_history_widget):
        try:
         reply = chat_with_rag(user_input, topic, spread, style, model_name)
        except Exception as e:
         reply = f"❌ 錯誤：{str(e)}"
        chat_history_widget.append((user_input, reply))
        return "", chat_history_widget

    msg.submit(
        respond_v2,
        [msg, topic_dropdown, spread_dropdown, style_dropdown, model_dropdown, chatbot],
        [msg, chatbot]
)


    demo.launch(debug=True)


In [ ]:
import gradio as gr

# 定義占卜主題、牌陣與占卜師風格
topic = gr.Radio(["感情", "工作", "學業", "人際", "人生目標"], label="選擇占卜主題")
topics = {
    "感情": "關於戀愛、曖昧、感情走向、復合、桃花運、婚姻等情感相關議題。",
    "工作": "針對職場、人際合作、職涯發展、升遷、跳槽、創業等職業相關議題。",
    "學業": "關於考試運、課業表現、升學、選系、留學等學習相關議題。",
    "人際": "朋友、家人、同事、人際互動、人緣、人際困擾等相關議題。",
    "人生目標": "針對人生方向、自我探索、目標訂定、階段任務等整體命運走向。"
}

spreads = ["單張占卜", "三張牌", "愛情十字陣", "五芒星陣"]
spreads = {
    "單張占卜": "抽一張牌，快速針對單一問題獲取命運訊息與建議，適合臨時直覺占卜。",
    "三張牌": "抽三張牌，分別代表『過去』『現在』『未來』，幫助看清問題脈絡與未來可能發展。",
    "愛情十字陣": "五張牌排列成十字，專門針對感情問題設計，洞察雙方現狀、潛意識、阻礙、建議與未來走向。",
    "五芒星陣": "五張牌排列成五芒星，適用於複雜問題，分別代表現況、障礙、潛能、建議、結果，全面分析局勢。"
}

style = gr.Radio(["冷靜分析型", "溫柔療癒型", "中二魔幻型", "搞笑活潑型"], label="選擇占卜師風格")
styles = {
    "冷靜分析型": "你是冷靜理性的命運分析者，以條理清晰、客觀務實方式解讀塔羅，避免誇張，重點在於實用建議。⚠️請**完全使用繁體中文**作答，禁用任何其他語言，每次回覆務必**直擊問題核心，解讀牌義並給出清楚、具體、實際命運建議**，避免冗長空泛。",
    "溫柔療癒型": "你是溫柔療癒的命運守護者，以溫暖柔和、安慰人心的語氣，細膩解讀塔羅，給予心靈安撫與希望。⚠️請**完全使用繁體中文**作答，禁用任何其他語言，每次回覆務必**直擊問題核心，解讀牌義並給出清楚、具體、實際命運建議**，避免冗長空泛。",
    "中二魔幻型": "你是神祕中二魔幻風格占卜師，語氣戲劇化誇張，充滿神祕符號與儀式感，讓人仿若置身魔法世界。⚠️請**完全使用繁體中文**作答，禁用任何其他語言，每次回覆務必**直擊問題核心，解讀牌義並給出清楚、具體、實際命運建議**，避免冗長空泛。",
    "搞笑活潑型": "你是幽默風趣、搞笑活潑的塔羅占卜師，用輕鬆詼諧方式解讀命運，帶給旅人快樂與正能量。⚠️請**完全使用繁體中文**作答，禁用任何其他語言，每次回覆務必**直擊問題核心，解讀牌義並給出清楚、具體、實際命運建議**，避免冗長空泛。"
}

# 聊天紀錄
chat_history = []

def chat_with_rag_v2(user_input, topic, spread, style):
    global chat_history

    # 取回資料庫內容
    docs = retriever.get_relevant_documents(user_input)
    retrieved_chunks = "\n\n".join([doc.page_content for doc in docs])

    # 套用不同風格的 system prompt
    selected_style_prompt = styles[style]

    final_system_prompt = (
        f"你是命運之塔的塔羅占卜大師，擅長以不同占卜師風格，"
        f"針對塔羅問題進行專業解讀。\n"
        f"本次占卜主題：「{topic}」，牌陣：「{spread}」，占卜師風格：「{style}」。\n\n"
        f"{selected_style_prompt}\n\n"
        "⚠️請**完全使用繁體中文**回覆，嚴禁出現任何英文、符號或其他語言，"
        "每次回覆務必依據資料庫塔羅牌義，給予清楚、具體、實用命運建議，搭配相應語氣及符號。"
    )

    final_prompt = f"""
🃏✦ 命運占卜儀式 啟動 ✦🃏

占卜主題：{topic}
牌陣方式：{spread}

依據命運之塔內典藏資料：
{retrieved_chunks}

🔮 旅人所求之命運問題為：
{user_input}

占卜師風格：{style}
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": final_system_prompt},
            {"role": "assistant", "content": "⚠️提醒：以下所有回覆皆需完全以繁體中文呈現，嚴禁英文及其他語言。"},
            {"role": "user", "content": final_prompt},
        ]
    )
    answer = response.choices[0].message.content

    chat_history.append((user_input, answer))
    return answer

with gr.Blocks() as demo:
    gr.Markdown("# 🃏命運之塔✦AI塔羅占卜大師")

    chatbot = gr.Chatbot(
        value=[
            ("命運之塔✦AI塔羅占卜大師",
             "🃏✦ 歡迎踏入命運之塔，旅人！✦🃏\n\n請在心中**默念你想詢問的問題三次**，讓心靈與命運之力共振…當心緒靜定，便可將你的疑問刻於下方之命運石板，我將為你揭示命運答案 🔮✨")
        ],
        height=500
    )

    msg = gr.Textbox(
        placeholder="將你任何人生相關的疑問刻於此處，讓我為你揭示命運之語——"
    )

    # ✅ 選單放在這裡一排
    with gr.Row():
        topic_dropdown = gr.Dropdown(
            ["不限", "感情", "工作", "學業", "人際", "人生目標"],
            label="🔮 占卜主題"
        )
        spread_dropdown = gr.Dropdown(
            ["單張占卜", "三張牌（過去/現在/未來）", "愛情十字陣", "五芒星陣"],
            label="🃏 牌陣"
        )
        style_dropdown = gr.Dropdown(
            ["冷靜分析型", "溫柔療癒型", "中二魔幻型", "搞笑活潑型"],
            label="✨ 占卜師風格"
        )

    # 假設你已經定義好 respond_v2 和 chat_with_rag_v2

    msg.submit(
        respond_v2,
        [msg, topic_dropdown, spread_dropdown, style_dropdown, chatbot],
        [msg, chatbot]
    )

    demo.launch(debug=True)